In [1]:

import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_FOLDS = 5
if os.path.exists('../../full_pipeline/olid_full_features.csv'):
    DATA_DIR = '../../full_pipeline'
    RESULTS_DIR = '.'
else:
    DATA_DIR = 'full_pipeline'
    RESULTS_DIR = 'New_Models_Baselines/01_Logistic_Regression'

os.makedirs(RESULTS_DIR, exist_ok=True)

csv_path = f'{DATA_DIR}/olid_full_features.csv'
df_full = pd.read_csv(csv_path)
df_accepted = df_full[df_full['quality_flag'] == 'ACCEPT'].copy().reset_index(drop=True)

meta_cols = ['image_path', 'image_name', 'class', 'crop', 'condition', 'quality_flag', 'mask_area_ratio', 'failure_reason']
feature_cols = [c for c in df_accepted.columns if c not in meta_cols]

df_folds = pd.read_csv(f'{DATA_DIR}/cv_fold_assignments.csv')
df_accepted = df_accepted.merge(df_folds[['image_path', 'fold']], on='image_path', how='left')

class FoldSafeScaler:
    def __init__(self):
        self._scaler = StandardScaler()
        self.is_fitted = False
    def fit_transform(self, X_train):
        arr = self._scaler.fit_transform(X_train.values)
        self.is_fitted = True
        return arr
    def transform(self, X_val):
        if not self.is_fitted:
            raise ValueError("FoldSafeScaler must be fitted first.")
        arr = self._scaler.transform(X_val.values)
        return arr

model = LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1, max_iter=1000)

fold_results = []

for fold_id in range(N_FOLDS):
    tr_idx = df_accepted[df_accepted['fold'] != fold_id].index
    va_idx = df_accepted[df_accepted['fold'] == fold_id].index
    y_tr = df_accepted.loc[tr_idx, 'class'].values
    y_va = df_accepted.loc[va_idx, 'class'].values
    X_tr = df_accepted.loc[tr_idx, feature_cols]
    X_va = df_accepted.loc[va_idx, feature_cols]

    sc = FoldSafeScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_va_sc = sc.transform(X_va)

    t0 = time.time()
    model.fit(X_tr_sc, y_tr)
    t_train = time.time() - t0

    t1 = time.time()
    y_pred = model.predict(X_va_sc)
    t_inf = time.time() - t1

    acc = accuracy_score(y_va, y_pred)
    mf1 = f1_score(y_va, y_pred, average='macro', zero_division=0)
    wf1 = f1_score(y_va, y_pred, average='weighted', zero_division=0)

    fold_results.append({
        'fold': fold_id,
        'accuracy': acc,
        'macro_f1': mf1,
        'weighted_f1': wf1,
        'train_time': t_train,
        'inference_time': t_inf
    })

df_fold = pd.DataFrame(fold_results)
df_fold.to_csv(f'{RESULTS_DIR}/logistic_regression_fold_results.csv', index=False)

summary = {
    'method': 'Logistic Regression',
    'mean_accuracy': df_fold['accuracy'].mean(),
    'std_accuracy': df_fold['accuracy'].std(),
    'mean_macro_f1': df_fold['macro_f1'].mean(),
    'std_macro_f1': df_fold['macro_f1'].std(),
    'mean_weighted_f1': df_fold['weighted_f1'].mean(),
    'std_weighted_f1': df_fold['weighted_f1'].std(),
    'mean_train_time': df_fold['train_time'].mean(),
    'mean_inference_time': df_fold['inference_time'].mean()
}
df_summary = pd.DataFrame([summary])
df_summary.to_csv(f'{RESULTS_DIR}/logistic_regression_summary.csv', index=False)

# Visualizations
plt.figure(figsize=(10, 6))
plt.errorbar([0], [summary['mean_macro_f1']], yerr=[summary['std_macro_f1']], fmt='o', capsize=5)
plt.title('Macro-F1 with Error Bars (Logistic Regression)')
plt.ylabel('Macro-F1')
plt.xticks([0], ['Logistic Regression'])
plt.savefig(f'{RESULTS_DIR}/macro_f1_plot.png')
plt.close()

plt.figure(figsize=(10, 6))
plt.errorbar([0], [summary['mean_accuracy']], yerr=[summary['std_accuracy']], fmt='o', capsize=5)
plt.title('Accuracy with Error Bars (Logistic Regression)')
plt.ylabel('Accuracy')
plt.xticks([0], ['Logistic Regression'])
plt.savefig(f'{RESULTS_DIR}/accuracy_plot.png')
plt.close()

plt.figure(figsize=(10, 6))
plt.bar(['Training Time', 'Inference Time'], [summary['mean_train_time'], summary['mean_inference_time']])
plt.title('Mean Training and Inference Latency (Logistic Regression)')
plt.ylabel('Time (seconds)')
plt.savefig(f'{RESULTS_DIR}/latency_plot.png')
plt.close()

# Comparison Table
historical = [
    {'Method': 'Linear SVM', 'Accuracy': 0.4855, 'Macro-F1': 0.4720},
    {'Method': 'RBF SVM', 'Accuracy': 0.4554, 'Macro-F1': 0.4307},
    {'Method': 'Random Forest', 'Accuracy': 0.5854, 'Macro-F1': 0.5499},
    {'Method': 'k-NN', 'Accuracy': 0.4876, 'Macro-F1': 0.4102},
    {'Method': 'Best standard WL (Feature-kNN h=2)', 'Accuracy': 0.4249, 'Macro-F1': 0.3911},
    {'Method': 'Logistic Regression', 'Accuracy': round(summary['mean_accuracy'], 4), 'Macro-F1': round(summary['mean_macro_f1'], 4)}
]
df_hist = pd.DataFrame(historical)

print("Concise Experiment Report")
print("=" * 60)
print(f"1. Dataset size: {len(df_accepted)} accepted OLID-I images")
print(f"2. Number of features: {len(feature_cols)}")
print(f"3. Number of classes: {df_accepted['class'].nunique()}")
print(f"4. CV protocol: {N_FOLDS}-fold stratified CV, training-fold-only fitting of scaler")
print(f"5. Logistic Regression hyperparameters: random_state={RANDOM_STATE}, class_weight='balanced', n_jobs=-1, max_iter=1000")
print("6. Fold-level results:")
print(df_fold.to_string(index=False))
print("\n7. Mean � SD results:")
print(f"   Accuracy:    {summary['mean_accuracy']:.4f} � {summary['std_accuracy']:.4f}")
print(f"   Macro-F1:    {summary['mean_macro_f1']:.4f} � {summary['std_macro_f1']:.4f}")
print(f"   Weighted-F1: {summary['mean_weighted_f1']:.4f} � {summary['std_weighted_f1']:.4f}")
print(f"8. Training time: {summary['mean_train_time']:.4f} seconds")
print(f"9. Inference time: {summary['mean_inference_time']:.4f} seconds")
print("\n10. Comparison with existing models:")
print(df_hist.to_string(index=False))
print("\n11. Warnings/errors: None observed during valid execution.")
print("12. Experiment successfully completed: YES")


Concise Experiment Report
1. Dataset size: 4387 accepted OLID-I images
2. Number of features: 38
3. Number of classes: 57
4. CV protocol: 5-fold stratified CV, training-fold-only fitting of scaler
5. Logistic Regression hyperparameters: random_state=42, class_weight='balanced', n_jobs=-1, max_iter=1000
6. Fold-level results:
 fold  accuracy  macro_f1  weighted_f1  train_time  inference_time
    0  0.452164  0.425268     0.458246    2.043085        0.000000
    1  0.421412  0.399269     0.421937    1.491616        0.002015
    2  0.416192  0.388636     0.417839    1.519099        0.002994
    3  0.490308  0.465696     0.489582    1.515324        0.000000
    4  0.445838  0.424329     0.445197    1.246037        0.000000

7. Mean � SD results:
   Accuracy:    0.4452 � 0.0295
   Macro-F1:    0.4206 � 0.0298
   Weighted-F1: 0.4466 � 0.0292
8. Training time: 1.5630 seconds
9. Inference time: 0.0010 seconds

10. Comparison with existing models:
                            Method  Accuracy  M